<a href="https://colab.research.google.com/github/tendolkarurja/Crime-Analytics-Dashboard/blob/main/Crimes_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv('/content/crime_dataset_india.csv', dayfirst=True)

In [ ]:
df

In [ ]:
df.drop('Time of Occurrence', axis =1, inplace = True)


In [ ]:
df['Date Reported'] = df['Date Reported'].str.split(' ').str[0]
df['Date of Occurrence'] = df['Date of Occurrence'].str.split(' ').str[0]
df['Date Case Closed'] = df['Date Case Closed'].str.split(' ').str[0]

In [ ]:
df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], dayfirst=False)
df['Date of Occurrence'] = df['Date of Occurrence'].dt.strftime('%d-%m-%Y')

In [ ]:
df['Date Reported'] = pd.to_datetime(df['Date Reported'], dayfirst=True).dt.date
df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], dayfirst=True).dt.date
df['Date Case Closed'] = pd.to_datetime(df['Date Case Closed'], dayfirst=True).dt.date


In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df['Crime Code'].value_counts()

In [ ]:
df['Crime Domain'].value_counts()

In [ ]:
df['Weapon Used'].value_counts()

In [ ]:
crime_per_city = df.City.value_counts().reset_index()
crime_per_city.columns = ['City', 'Total Crimes']
crime_per_city

In [ ]:
plt.figure(figsize=(15, 8))
sns.barplot(data=crime_per_city, x='City', y='Total Crimes', palette='viridis')
plt.title('Total Crimes Reported per City', fontsize=16, fontweight='bold')
plt.xlabel('City Name', fontsize=12)
plt.ylabel('Total Crimes Observed', fontsize=12)

plt.xticks(rotation=-45)
plt.show()

In [ ]:
df['Victim Gender'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.kdeplot(
    data=df,
    x='Victim Age',
    hue='Victim Gender',
)

plt.title('Victim Age Distribution by Gender')
plt.show()

In [ ]:
df['Crime Description'].value_counts()

In [ ]:
other_crimes = df[df['Weapon Used'] == 'Other'].groupby('Crime Description')
other_crimes

In [ ]:
other_crimes['Crime Description'].value_counts()

In [ ]:
df['Crime Description'].value_counts().sum()

In [ ]:
other_crimes['Crime Description'].value_counts()

In [ ]:
resolved = df.dropna(subset=['Date Case Closed'])
resolved.drop('Case Closed', axis = 1, inplace = True)
resolved

In [ ]:
resolved['Time to Resolve'] = (pd.to_datetime(resolved['Date Case Closed']) - pd.to_datetime(resolved['Date Reported'])).dt.days

In [ ]:
resolved

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure you are using the 'resolved' subset (where Case Closed is not null)
plt.figure(figsize=(10, 6))

# Use a scatter plot to see the distribution
plt.scatter(resolved['Police Deployed'], resolved['Time to Resolve'], alpha=0.5, color='teal')

plt.title('Correlation: Police Deployment vs. Resolution Speed')
plt.xlabel('Number of Police Deployed')
plt.ylabel('Days to Resolve Case')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
city_resourcing = df.groupby('City')['Police Deployed'].sum().sort_values(ascending=False).reset_index()

# 2. Plotting
plt.figure(figsize=(15, 8))
sns.barplot(data=city_resourcing, x='City', y='Police Deployed', palette='viridis')

plt.title('Total Police Deployment per City (Sorted)', fontsize=16)
plt.xlabel('City', fontsize=12)
plt.ylabel('Police Deployed', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
time_per_city = resolved.groupby('City')['Time to Resolve'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(15, 8))
sns.barplot(data=time_per_city, x='City', y='Time to Resolve', palette='viridis')

plt.title('Average resolution time per City', fontsize=16)
plt.xlabel('City', fontsize=12)
plt.ylabel('Avg. Resolution Time', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
time_per_city = resolved.groupby('City')['Time to Resolve'].median().sort_values(ascending=False).reset_index()

plt.figure(figsize=(15, 8))
sns.barplot(data=time_per_city, x='City', y='Time to Resolve', palette='viridis')

plt.title('Median resolution time per City', fontsize=16)
plt.xlabel('City', fontsize=12)
plt.ylabel('Median Resolution Time', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
other_crimes = df[df['Crime Domain'] == 'Other Crime']
sns.countplot(data=other_crimes, x='Victim Gender', palette='magma')
plt.title('Gender Distribution within "Other Crime" Category')
plt.show()

In [ ]:
sns.countplot(data=resolved, x='Victim Gender', palette='magma')
plt.title('Gender Distribution within "Other Crime" Category')
plt.show()

In [ ]:
df['Time to Report'] = (pd.to_datetime(df['Date Reported']) -pd.to_datetime(df['Date of Occurrence'])).dt.days
city_reporting = df.groupby('City')['Time to Report'].median().sort_values(ascending=False).reset_index()

plt.figure(figsize=(15, 8))
sns.barplot(data=city_reporting, x='City', y='Time to Report', palette='viridis')

plt.title('Mean Reporting time per City', fontsize=16)
plt.xlabel('City', fontsize=12)
plt.ylabel('Median Reporting Time', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
pivot_df = df.groupby(['City', 'Case Closed']).size().unstack(fill_value=0)
pivot_df['Total'] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values('Total', ascending=False).drop(columns='Total')

# 3. Plotting
plt.figure(figsize=(15, 8))
# Assigning clear colors: Green for 'Yes' (Resolved), Red for 'No' (Unresolved)
pivot_df.plot(kind='bar', stacked=True, color=['#e76f51', '#2a9d8f'], ax=plt.gca())

plt.title('Case Resolution Backlog', fontsize=16, fontweight='bold')
plt.xlabel('City', fontsize=12)
plt.ylabel('Total Crimes Reported', fontsize=12)
plt.legend(title="Resolved?", labels=['No', 'Yes'])
plt.xticks(rotation=45)
plt.show()

In [ ]:
import matplotlib.cm as cm

unresolved_df = df[df['Case Closed'] == 'No']
domain_counts = unresolved_df['Crime Domain'].value_counts()

colors = cm.coolwarm(np.linspace(0, 1, len(domain_counts)))

# 3. Plotting
plt.figure(figsize=(10, 8))

plt.pie(domain_counts,
        labels=domain_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        explode=[0.1 if i == 0 else 0 for i in range(len(domain_counts))],
        shadow=True,
        textprops={'fontsize': 12})

plt.title('Distribution of Unresolved Cases by Crime Domain', fontsize=15, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
df['Weapon Used'].fillna('Unspecified', inplace = True)

In [ ]:
df

In [ ]:
df.drop(columns=['Report Number', 'Crime Code'], inplace=True)

In [ ]:
df

In [ ]:
resolved.drop(columns = ['Report Number', 'Crime Code'], inplace = True)

In [ ]:
resolved

In [ ]:
resolved['Weapon Used'].fillna('Unspecified', inplace = True)

In [ ]:
resolved

In [ ]:
df.isna().sum()

In [ ]:
unresolved = df[df['Case Closed'] == 'No']
weapon_counts = unresolved['Weapon Used'].value_counts()

plt.figure(figsize=(10, 10))
plt.pie(weapon_counts,
        labels=weapon_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette('coolwarm', len(weapon_counts)))
plt.title('Weapon Distribution in Unresolved Cases', fontsize=15, fontweight='bold')
plt.show()

In [ ]:
df['Case Closed'].value_counts()

In [ ]:
resolution_counts = df['Case Closed'].value_counts()

plt.figure(figsize=(8, 8))
colors = ['#e76f51', '#2a9d8f']

plt.pie(resolution_counts,
        labels=['Unresolved (No)', 'Resolved (Yes)'],
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        explode=(0.05, 0), # Explode the 'Unresolved' slice for emphasis
        shadow=True,
        textprops={'fontsize': 14})

plt.title('Overall Case Resolution Distribution (2020-2024)', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
city_weapons = df.groupby('City')['Weapon Used'].agg(lambda x: x.mode()[0]).reset_index()

weapon_wins = city_weapons['Weapon Used'].value_counts()

plt.figure(figsize=(12, 7))
sns.barplot(x=weapon_wins.index, y=weapon_wins.values, palette='viridis')

plt.title('Which Weapon "Dominates" the Most Cities?', fontsize=16, fontweight='bold')
plt.xlabel('Weapon Category', fontsize=12)
plt.ylabel('Number of Cities where this is the #1 Weapon', fontsize=12)
plt.xticks(rotation=45)

for i, v in enumerate(weapon_wins.values):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('Crimes.csv', index=False)
resolved.to_csv('Resolved crimes.csv', index = False)